# QuRe Exp1 · PFM sinh trực tiếp query token reference

Chạy **Run All** trên Kaggle với GPU, Internet và FashionIQ đã giải nén. Nếu gắn nhiều bộ FashionIQ, đặt `DATA_ROOT` ở cell cấu hình. Notebook nạp module và trainer từ commit ghi trong `RUNNER_SHA`. Mặc định dùng 30 epoch, batch 16 và seed 1000.

PFM lấy trung bình chuỗi token ảnh, suy ra Gaussian latent 64 chiều và tham số của 10 planar flow. Decoder sinh 32 token 768 chiều cho Q-Former reference. Caption và chuỗi token ảnh đầy đủ cũng đi vào Q-Former. Nhánh target dùng bộ query token pretrained. Hai nhánh dùng chung trọng số Q-Former.

Train dùng contrastive loss trong batch cộng KL của flow. Khi validation, PFM dùng mean của Gaussian trước flow và xếp hạng ảnh bằng điểm retrieval.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import shutil
import hashlib

RUNS = ['direct_planar_image']
DATA_ROOT = None  # Path('/kaggle/input/.../fashionIQ') nếu tự tìm không được
CHECKPOINT = None  # Checkpoint BLIP-2 pretrained, không phải checkpoint đã train
RESUME_FROM = {}  # Ví dụ {'direct_planar_image': '/kaggle/input/run/last.pt'}
SEED = 1000
BATCH_SIZE = 16
EPOCHS = 30
KL_WEIGHT = 0.001
KL_WARMUP_EPOCHS = 5
RUN_TAG = 's1'  # Đổi tag để bắt đầu lượt mới với cấu hình khác
WORK = Path('/kaggle/working')
EXPERIMENT_SHA = 'b36607df9d7813167df9cf2ef1d1de605938f1f2'
BACKBONE_SHA = 'c1c7f6b62fe6d556b9554b6c7cae111f71f67917'
RUNNER_SHA = '2be8e012314dbfb4f46b406e6a6e633a5e8a055a'
RUNNER = WORK / 'exp1_direct_runner_source'
PACKAGE = RUNNER / 'experiments' / 'exp1_direct_query_pfm'
BACKBONE = WORK / 'qure_main_backbone'
CODE = WORK / 'exp1_direct_query_pfm_20261119'
CODE.mkdir(exist_ok=True)
VALID = {'direct_planar_image'}
assert RUNS and set(RUNS) <= VALID and BATCH_SIZE >= 2


## Cài thư viện và lấy source đã khóa commit

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle settings.")
subprocess.run([sys.executable, "-m", "pip", "install", "--prefer-binary", "--only-binary=tokenizers",
                "transformers==4.46.3", "tokenizers==0.20.3",
                "timm==0.9.16", "fairscale", "omegaconf", "iopath"], check=True)

# Verify dependencies in a fresh process, as used by the training runner.
subprocess.run([sys.executable, "-c", """
import sys, torch, transformers, tokenizers
from transformers.modeling_utils import (
    PreTrainedModel, apply_chunking_to_forward,
    find_pruneable_heads_and_indices, prune_linear_layer,
)
assert transformers.__version__ == "4.46.3"
assert tokenizers.__version__ == "0.20.3"
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__, "| tokenizers:", tokenizers.__version__)
"""], check=True)

def checkout(repo, sha, destination):
    if not (destination / ".git").is_dir():
        destination.mkdir(exist_ok=True)
        subprocess.run(["git", "init", "-q", str(destination)], check=True)
    probe = subprocess.run(["git", "-C", str(destination), "rev-parse", "--verify", "HEAD"], capture_output=True, text=True)
    if probe.stdout.strip() != sha:
        subprocess.run(["git", "-C", str(destination), "fetch", "--depth=1", repo, sha], check=True)
        subprocess.run(["git", "-C", str(destination), "checkout", "--detach", sha], check=True)
    print(destination.name, subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"], text=True).strip())

checkout("https://github.com/fast2104/QuRe.git", RUNNER_SHA, RUNNER)
checkout("https://github.com/fast2104/QuRe.git", BACKBONE_SHA, BACKBONE)

## Dataset và checkpoint pretrained

In [ ]:
import shutil
inputs = Path("/kaggle/input")
if DATA_ROOT is None:
    matches = [p.parent.parent for p in inputs.rglob("cap.dress.train.json")
               if (p.parent.parent / "images").is_dir() and (p.parent.parent / "image_splits").is_dir()]
    if len(matches) != 1:
        raise ValueError(f"Set DATA_ROOT to the extracted FashionIQ directory. Candidates: {matches}")
    DATA_ROOT = matches[0]
DATA_ROOT = Path(DATA_ROOT)
for category in ("dress", "shirt", "toptee"):
    for split in ("train", "val"):
        annotations = DATA_ROOT / "captions" / f"cap.{category}.{split}.json"
        sample = json.loads(annotations.read_text())[0]
        for key in ("candidate", "target"):
            image = DATA_ROOT / "images" / (sample[key] + ".png")
            if not image.is_file():
                raise FileNotFoundError(image)
    assert (DATA_ROOT / "image_splits" / f"split.{category}.val.json").is_file()

name = "blip2_pretrained_vitL.pth"
if CHECKPOINT is None:
    CHECKPOINT = next(inputs.rglob(name), WORK / name)
CHECKPOINT = Path(CHECKPOINT).resolve()
if not CHECKPOINT.is_file():
    if CHECKPOINT != WORK / name:
        raise FileNotFoundError(CHECKPOINT)
    base_url = "https://storage.googleapis.com/sfr-vision-language-research/LAVIS/models/BLIP2/"
    temporary = WORK / (name + ".part")
    subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(temporary), base_url + name], check=True)
    temporary.replace(CHECKPOINT)
if CHECKPOINT.name != name:
    raise ValueError(f"Expected checkpoint filename {name}")
os.environ["QURe_CHECKPOINT_ROOT"] = str(CHECKPOINT.parent)
print("FashionIQ:", DATA_ROOT, "| BLIP-2:", CHECKPOINT)

## Chuẩn bị backbone

In [ ]:
def original(rel):
    return subprocess.check_output(["git", "-C", str(BACKBONE), "show", f"HEAD:{rel}"], text=True)

model_init = "lavis/models/__init__.py"
source = original(model_init)
loaders = source[source.index("def load_model("):source.index("class ModelZoo")]
imports = """import logging
import torch
from omegaconf import OmegaConf
from lavis.common.registry import registry
from lavis.models.base_model import BaseModel
from lavis.processors import BaseProcessor
from lavis.models.blip2_models.blip2_qure import Blip2QuRe
"""
(BACKBONE / model_init).write_text(imports + "\n" + loaders)
(BACKBONE / "lavis/processors/__init__.py").write_text(
    "from lavis.processors.base_processor import BaseProcessor\nfrom lavis.processors import blip_processors\n")
(BACKBONE / "data/__init__.py").write_text("# Retrieval runner imports only data.utils.\n")
yaml_path = "lavis/configs/models/blip2/blip2_pretrain_vitL.yaml"
import re
yaml_text, count = re.subn(r"(?m)^  pretrained:.*$", lambda _: "  pretrained: " + json.dumps(str(CHECKPOINT)), original(yaml_path))
assert count == 1
(BACKBONE / yaml_path).write_text(yaml_text)

## Mã nguồn của lượt train
Sao chép model, trainer và hai bài kiểm tra từ commit đã chọn vào thư mục chạy.

In [ ]:
for filename in ("model.py", "data.py", "pfm_variants.py", "train_pfm.py", "verify_variants.py", "preflight.py"):
    shutil.copyfile(PACKAGE / filename, CODE / filename)
module_source = (CODE / "pfm_variants.py").read_text(encoding="utf-8")
trainer = (CODE / "train_pfm.py").read_text(encoding="utf-8")

## Kiểm tra trước khi train
Kiểm tra đầu ra, gradient từ token về posterior/flow/decoder, điều kiện ảnh, train ngẫu nhiên, eval xác định, state reload và Jacobian. Kiểm tra GPU với Exp1 thật xác nhận thay đổi token gốc không làm đổi query reference và gradient reference tới token gốc là None. Sau đó kiểm tra loss đầy đủ và backward.

In [ ]:
subprocess.run([sys.executable, str(CODE / "verify_variants.py")], cwd=CODE, check=True)

## Huấn luyện
Checkpoint last.pt lưu mỗi epoch và best.pt theo Avg validation tốt nhất. Resume cần checkpoint cùng phiên bản/cấu hình. Với session mới, gắn outputs cũ vào Kaggle Input và khai báo RESUME_FROM. Cuối buổi dùng Save Version với outputs.

In [ ]:
template = json.loads((PACKAGE / 'config.json').read_text())
config_paths = {}
output_paths = {}
source_digest = hashlib.sha256((module_source + trainer).encode()).hexdigest()
for variant in RUNS:
    output = WORK / f'exp1_suite_{variant}_seed{SEED}_{RUN_TAG}'
    pfm = dict(variant=variant, hidden=64, num_flows=10,
               kl_weight=KL_WEIGHT, kl_warmup_epochs=KL_WARMUP_EPOCHS)
    config = dict(template, seed=SEED, batch_size=BATCH_SIZE, epochs=EPOCHS,
                  output_dir=str(output), pfm=pfm, source_sha256=source_digest,
                  experiment_sha=EXPERIMENT_SHA, backbone_sha=BACKBONE_SHA)
    config_path = CODE / f'config_{variant}.json'
    config_path.write_text(json.dumps(config, indent=2))
    config_paths[variant] = config_path
    output_paths[variant] = output
    resume = Path(RESUME_FROM[variant]) if variant in RESUME_FROM else output / 'last.pt'
    if variant in RESUME_FROM and not resume.is_file():
        raise FileNotFoundError(resume)
    if resume.is_file():
        saved = json.loads((resume.parent / 'config.json').read_text())
        expected = dict(config)
        saved.pop('output_dir', None)
        expected.pop('output_dir', None)
        if saved != expected:
            raise ValueError(f'Checkpoint/config mismatch: {resume}. Use matching settings or a new RUN_TAG.')
        output.mkdir(exist_ok=True)
        if resume.parent.resolve() != output.resolve():
            for filename in ('history.json', 'run_summary.json', 'latest_metrics.json', 'best.pt'):
                source = resume.parent / filename
                if source.exists():
                    shutil.copy2(source, output / filename)
    elif (output / 'history.json').exists():
        raise RuntimeError(f'History exists without last.pt in {output}; select a new RUN_TAG.')
    subprocess.run([sys.executable, str(CODE / 'preflight.py'), str(config_path),
                    str(BACKBONE), str(DATA_ROOT)], cwd=CODE, check=True)
    output.mkdir(exist_ok=True)
    for filename in ('pfm_variants.py', 'train_pfm.py', 'model.py', 'data.py'):
        shutil.copy2(CODE / filename, output / filename)
    command = [sys.executable, '-u', str(CODE / 'train_pfm.py'),
               '--config', str(config_path), '--qure-root', str(BACKBONE),
               '--fashioniq-root', str(DATA_ROOT), '--output-dir', str(output)]
    if resume.is_file():
        command += ['--resume', str(resume)]
    with (output / 'console.log').open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=CODE, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)


## Kết quả validation
Đọc epoch cuối và epoch có Avg recall cao nhất từ `history.json` của chính lượt chạy.

In [ ]:
import pandas as pd
rows = []
for variant, output in output_paths.items():
    history = json.loads((output / 'history.json').read_text())
    evaluated = [r for r in history if 'average_recall' in r]
    if evaluated:
        last = evaluated[-1]
        best = max(evaluated, key=lambda r: r['average_recall'])
        rows.append(dict(variant=variant, last_epoch=last['epoch'],
                         R10=last['average_recall_at10'], R50=last['average_recall_at50'],
                         Avg=last['average_recall'], best_epoch=best['epoch'], best_Avg=best['average_recall']))
summary = pd.DataFrame(rows)
display(summary.round(3))
summary.to_csv(CODE / 'results_summary.csv', index=False)
